In [1]:
from evo.tools import file_interface
from evo.core import sync
from evo.core import metrics
from evo.core.geometry import GeometryException
import matplotlib.pyplot as plt
from evo.core.units import Unit
from rosbags.rosbag1 import Reader
from numpy import mean
import pprint, os

In [8]:
def get_speedup(maps_nr:int, bag:str) -> int:
    wall_duration = maps_nr*30
    with Reader(bag) as reader:
        ros_duration = reader.duration
    return round(ros_duration/10**9/wall_duration)

def get_map_nr(path:str):
    # path: main path of the bag
    return len([file for file in os.listdir(f'{path}{'/' if path else ''}Maps') if file.endswith('Map.png') and file!='Map.png'])

def get_incremental_APE(bag:str, maps_nr:int, map_freq:int=30):
    '''
         bag: name of bag file
     maps_nr: number of partial maps
    map_freq: wall time frequencie to capture map
    '''
    with Reader(bag) as reader:
        traj_est = file_interface.read_bag_trajectory(reader,'/odom')
        traj_ref = file_interface.read_bag_trajectory(reader,'/base_pose_ground_truth')
    traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est)
    try:
        traj_est.align(traj_ref,correct_scale=True)
    except GeometryException:
        print(f'RUN ROTTA: {bag}')
        return None
    ate_metric = metrics.APE(metrics.PoseRelation.translation_part)
    ate_metric.process_data((traj_ref, traj_est))
    are_metric = metrics.APE(metrics.PoseRelation.rotation_part)
    are_metric.process_data((traj_ref, traj_est))
    ape_metric = metrics.APE(metrics.PoseRelation.full_transformation)
    ape_metric.process_data((traj_ref, traj_est))
    ts = traj_est.timestamps
    freq = mean([ts[i]-ts[i-1] for i in range(1,len(ts))])
    #print(f'freq: {freq}')
    step = int(map_freq/freq)*get_speedup(maps_nr,bag)
    return {
        'ATE': ate_metric.error[::step],
        'ARE': are_metric.error[::step],
        'APE': ape_metric.error[::step],
    }

def make_APE_log(path:str, APE:dict):
    '''
    path: main path of the bag
     APE: {'ATE':[x1,...,xn],'ARE':[y1,...,yn]} (n=maps_number)
    ''' 
    n = get_map_nr(path)
    for i in range(n):
        with open(f'{path}{'/' if path else ''}Maps/{i}APE.yaml','w') as file:
            file.write(f'ate: {APE['ATE'][i]}\nare: {APE['ARE'][i]}\nape: {APE['APE'][i]}')

In [14]:
path = 'ros-exploration-config/output'
names = os.listdir(path)
for name in names:
    main_path = f'{path}/{name}/run1'
    bag = f'{main_path}/{name}.bag'
    APE = get_incremental_APE(bag, get_map_nr(main_path))
    if APE: 
        print(name)
        make_APE_log(main_path, APE)

0510025536_Layout1_PLAN4
0510025537_Layout1_PLAN3
0510030965_A-40
0510030966_A-40
0510030967_A-40
RUN ROTTA: ros-exploration-config/output/0510030968_A-40/run1/0510030968_A-40.bag
RUN ROTTA: ros-exploration-config/output/0510032192_A_40_1_103/run1/0510032192_A_40_1_103.bag
0510032192_A_40_1_203
RUN ROTTA: ros-exploration-config/output/0510032194_A_40_1_102/run1/0510032194_A_40_1_102.bag
RUN ROTTA: ros-exploration-config/output/0510032194_A_40_1_202/run1/0510032194_A_40_1_202.bag
0510032216_A-40
0510032217_A-40
0510032218_A-40
RUN ROTTA: ros-exploration-config/output/0510032258_A_40_1_105/run1/0510032258_A_40_1_105.bag
0510032259_A_40_1_101
0510032260_A_40_1_102
0510032266_A_40_1_103
0510032267_A_40_1_101
RUN ROTTA: ros-exploration-config/output/0510032268_A_40_1_102/run1/0510032268_A_40_1_102.bag
0510032270_A_40_1_104
RUN ROTTA: ros-exploration-config/output/0510032271_A_40_1_105/run1/0510032271_A_40_1_105.bag
0510034689_Layout1
0510034690_Layout1
RUN ROTTA: ros-exploration-config/outp